In [4]:
# =========================================================
# ADVANCED XOR VISUALIZATION USING PYTORCH
# IMPROVED VISUALS
# =========================================================

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# =========================================================
# DEVICE CONFIGURATION
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

# =========================================================
# XOR DATASET
# =========================================================

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y = np.array([[0], [1], [1], [0]], dtype=np.float32)

X_tensor = torch.tensor(X).to(device)
y_tensor = torch.tensor(y).to(device)

# =========================================================
# IMAGE 1 — XOR DATASET  (smaller, centered, annotated)
# =========================================================
# Key fix: push points INWARD (-0.2 offset) and add "why not linearly
# separable" lines so the message is immediately clear.

fig, ax = plt.subplots(figsize=(5.5, 5.5))   # smaller figure

COLORS = {0: '#E74C3C', 1: '#2980B9'}
LABELS = {0: 'Class 0  (XOR = 0)', 1: 'Class 1  (XOR = 1)'}
MARKERS = {0: 'o', 1: 's'}

plotted = set()
for i, (xi, yi) in enumerate(zip(X, y.ravel())):
    lbl = LABELS[int(yi)] if int(yi) not in plotted else ""
    ax.scatter(xi[0], xi[1],
               s=350, marker=MARKERS[int(yi)],
               color=COLORS[int(yi)],
               edgecolors='white', linewidths=1.8,
               zorder=5, label=lbl)
    ax.annotate(f"({int(xi[0])}, {int(xi[1])})\nXOR={int(yi)}",
                xy=(xi[0], xi[1]),
                xytext=(xi[0] + 0.11, xi[1] + 0.11),
                fontsize=10, fontweight='bold',
                color=COLORS[int(yi)],
                arrowprops=dict(arrowstyle="-", color='gray', lw=0.8))
    plotted.add(int(yi))

# Show TWO attempted linear separators — both fail — conveying non-linearity
x_line = np.linspace(-0.5, 1.5, 200)
ax.plot(x_line,  0.5 * np.ones_like(x_line), '--', color='gray',
        alpha=0.55, lw=1.4, label='Failed linear cuts')
ax.plot(x_line, -x_line + 1,                  '--', color='gray',
        alpha=0.55, lw=1.4)
ax.plot(x_line,  x_line,                       '--', color='gray',
        alpha=0.55, lw=1.4)

# Hatched shading for "same class" pairs to show the problem visually
ax.fill_between([-0.4, 0.5], [-0.4, -0.4], [0.5, 0.5],
                color='#E74C3C', alpha=0.07)
ax.fill_between([0.5, 1.4], [0.5, 0.5], [1.4, 1.4],
                color='#E74C3C', alpha=0.07)
ax.fill_between([-0.4, 0.5], [0.5, 0.5], [1.4, 1.4],
                color='#2980B9', alpha=0.07)
ax.fill_between([0.5, 1.4], [-0.4, -0.4], [0.5, 0.5],
                color='#2980B9', alpha=0.07)

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xticks([0, 1]);  ax.set_yticks([0, 1])
ax.tick_params(labelsize=11)
ax.set_xlabel("Input x₁", fontsize=12, fontweight='bold')
ax.set_ylabel("Input x₂", fontsize=12, fontweight='bold')
ax.set_title("XOR Problem — Not Linearly Separable", fontsize=13, fontweight='bold', pad=12)
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend(fontsize=10, loc='lower right', framealpha=0.9)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig("xor_dataset_better.png", dpi=180, bbox_inches='tight')
plt.close()
print("Saved xor_dataset_better.png")

# =========================================================
# BUILD & TRAIN MODEL
# =========================================================

class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)
        self.output = nn.Linear(4, 1)
        self.relu   = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.output(self.relu(self.hidden(x))))

    def hidden_features(self, x):
        return self.relu(self.hidden(x))

torch.manual_seed(42)
model = XORNet().to(device)

criterion  = nn.BCELoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=0.01)

EPOCHS = 3000
losses = []

for epoch in range(EPOCHS):
    pred = model(X_tensor)
    loss = criterion(pred, y_tensor)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 500 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss: {loss.item():.6f}")

# =========================================================
# IMAGE 2 — TRAINING LOSS
# =========================================================

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(losses, lw=2, color='#2980B9')
ax.fill_between(range(EPOCHS), losses, alpha=0.15, color='#2980B9')
ax.set_title("Training Loss", fontsize=14, fontweight='bold')
ax.set_xlabel("Epoch", fontsize=12); ax.set_ylabel("BCE Loss", fontsize=12)
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig("xor_training_loss.png", dpi=180, bbox_inches='tight')
plt.close()
print("Saved xor_training_loss.png")

# =========================================================
# EXTRACT HIDDEN FEATURES & OUTPUT WEIGHTS
# =========================================================

model.eval()
with torch.no_grad():
    hidden_features = model.hidden_features(X_tensor).cpu().numpy()
    W_out = model.output.weight.cpu().numpy()[0]   # shape (4,)
    b_out = float(model.output.bias.cpu().numpy()[0])

print("\nHidden features (4D):\n", hidden_features)
print("Output weights:", W_out, "  bias:", b_out)

# =========================================================
# IMAGE 3 — ACTUAL HYPERPLANE IN 3D HIDDEN SPACE
# =========================================================
# We visualise the first 3 hidden neurons. The 4th is fixed at its
# mean value across the 4 data points to derive an effective bias,
# giving the TRUE decision boundary from the trained weights.

h4_mean   = hidden_features[:, 3].mean()
b_eff     = b_out + W_out[3] * h4_mean   # effective bias in 3D subspace
w0, w1, w2 = W_out[0], W_out[1], W_out[2]

# Class colours
point_colors = ['#E74C3C' if int(yi) == 0 else '#2980B9' for yi in y.ravel()]
point_markers = ['o' if int(yi) == 0 else '^' for yi in y.ravel()]

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

# Scatter the 4 data points
for i in range(4):
    ax.scatter(hidden_features[i, 0],
               hidden_features[i, 1],
               hidden_features[i, 2],
               s=280,
               c=point_colors[i],
               marker=point_markers[i],
               edgecolors='white', linewidths=1.5,
               zorder=6,
               label=('Class 0' if int(y[i][0]) == 0 else 'Class 1')
                      if i < 2 else "")
    ax.text(hidden_features[i, 0],
            hidden_features[i, 1],
            hidden_features[i, 2] + 0.05,
            f" ({int(X[i,0])},{int(X[i,1])})",
            fontsize=9, color=point_colors[i], fontweight='bold')

# ── Build the ACTUAL decision hyperplane ──────────────────────────────
# Boundary condition:  w0*h1 + w1*h2 + w2*h3 + b_eff = 0
# → h3 = -(w0*h1 + w1*h2 + b_eff) / w2   (if w2 ≠ 0)

pad = 0.4
h1_vals = np.linspace(hidden_features[:, 0].min() - pad,
                      hidden_features[:, 0].max() + pad, 30)
h2_vals = np.linspace(hidden_features[:, 1].min() - pad,
                      hidden_features[:, 1].max() + pad, 30)
HH1, HH2 = np.meshgrid(h1_vals, h2_vals)

if abs(w2) > 1e-6:
    HH3 = -(w0 * HH1 + w1 * HH2 + b_eff) / w2
else:
    # Fall back: solve for whichever weight is largest
    if abs(w1) > abs(w0):
        HH3 = HH2          # just for shape; relabel
        HH2 = -(w0 * HH1 + w2 * HH3 + b_eff) / w1
    else:
        HH1 = -(w1 * HH2 + w2 * HH3 + b_eff) / w0

# Clip the plane to a sensible z-range so it doesn't fly off screen
z_lo = hidden_features[:, 2].min() - pad * 2
z_hi = hidden_features[:, 2].max() + pad * 2
HH3  = np.clip(HH3, z_lo, z_hi)

surf = ax.plot_surface(HH1, HH2, HH3,
                       alpha=0.35, color='#27AE60',
                       rstride=1, cstride=1,
                       linewidth=0, antialiased=True)

# Fake legend patch for the plane
plane_patch = mpatches.Patch(color='#27AE60', alpha=0.5,
                              label='Decision hyperplane\n(output layer boundary)')

# Draw normal vector of the plane from its centre
cx = HH1.mean(); cy = HH2.mean(); cz = HH3.mean()
norm = np.array([w0, w1, w2])
norm = norm / (np.linalg.norm(norm) + 1e-9) * 0.6
ax.quiver(cx, cy, cz, norm[0], norm[1], norm[2],
          color='#27AE60', linewidth=2, arrow_length_ratio=0.3)

handles, lbls = ax.get_legend_handles_labels()
# deduplicate
seen = {}
for h, l in zip(handles, lbls):
    if l not in seen: seen[l] = h
seen['Decision hyperplane\n(output layer boundary)'] = plane_patch
ax.legend(seen.values(), seen.keys(), fontsize=9, loc='upper left')

ax.set_title("Hidden Space (3 of 4 neurons) — Actual Decision Hyperplane",
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel("Hidden Neuron 1", fontsize=10)
ax.set_ylabel("Hidden Neuron 2", fontsize=10)
ax.set_zlabel("Hidden Neuron 3", fontsize=10)

plt.tight_layout()
plt.savefig("xor_hidden_space_hyperplane.png", dpi=180, bbox_inches='tight')
plt.close()
print("Saved xor_hidden_space_hyperplane.png")

# =========================================================
# IMAGE 4 — ACTUAL vs PREDICTED
# =========================================================

with torch.no_grad():
    preds_raw = model(X_tensor).cpu().numpy().ravel()

binary_preds = (preds_raw > 0.5).astype(int)
labels = ['(0,0)', '(0,1)', '(1,0)', '(1,1)']
x_pos  = np.arange(4)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x_pos - 0.18, y.ravel(), 0.32, label='Actual',    color='#E74C3C', alpha=0.85)
ax.bar(x_pos + 0.18, preds_raw, 0.32, label='Predicted', color='#2980B9', alpha=0.85)
ax.axhline(0.5, color='gray', lw=1.2, ls='--', label='Decision threshold (0.5)')
ax.set_xticks(x_pos); ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(-0.1, 1.25)
ax.set_title("Actual vs Predicted XOR Outputs", fontsize=13, fontweight='bold')
ax.set_xlabel("Input (x₁, x₂)", fontsize=12)
ax.set_ylabel("Output probability", fontsize=12)
ax.legend(fontsize=11); ax.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig("xor_actual_vs_predicted.png", dpi=180, bbox_inches='tight')
plt.close()
print("Saved xor_actual_vs_predicted.png")

# =========================================================
# PREDICTIONS + ACCURACY
# =========================================================

print("\n================ PREDICTIONS ================\n")
for i in range(4):
    print(f"Input: {X[i]}  →  Prob: {preds_raw[i]:.4f}  →  Binary: {binary_preds[i]}")

accuracy = np.mean(binary_preds == y.ravel())
print(f"\nModel Accuracy: {accuracy*100:.2f}%")

Using Device: cuda
Saved xor_dataset_better.png
Epoch [500/3000]  Loss: 0.014657
Epoch [1000/3000]  Loss: 0.002985
Epoch [1500/3000]  Loss: 0.001221
Epoch [2000/3000]  Loss: 0.000639
Epoch [2500/3000]  Loss: 0.000378
Epoch [3000/3000]  Loss: 0.000241
Saved xor_training_loss.png

Hidden features (4D):
 [[2.3811533e+00 1.8340077e-06 3.0672297e+00 2.3526497e-01]
 [1.0491978e+00 3.9428561e+00 0.0000000e+00 5.0693579e+00]
 [4.4715958e+00 0.0000000e+00 0.0000000e+00 9.1955066e-05]
 [3.1396403e+00 0.0000000e+00 0.0000000e+00 4.8341846e+00]]
Output weights: [ 1.6026813  5.2320833 -4.158383  -2.9527216]   bias: 1.0096116065979004
Saved xor_hidden_space_hyperplane.png
Saved xor_actual_vs_predicted.png

================ PREDICTIONS ================

Input: [0. 0.]  →  Prob: 0.0002  →  Binary: 0
Input: [0. 1.]  →  Prob: 0.9998  →  Binary: 1
Input: [1. 0.]  →  Prob: 0.9997  →  Binary: 1
Input: [1. 1.]  →  Prob: 0.0003  →  Binary: 0

Model Accuracy: 100.00%
